In [1]:
from datasets import load_dataset

csqa_data = load_dataset("tau/commonsense_qa")

print(csqa_data)

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 1140/1140 [00:00<00:00, 316907.91 examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
        num_rows: 9741
    })
    validation: Dataset({
        features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
        num_rows: 1221
    })
    test: Dataset({
        features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
        num_rows: 1140
    })
})


In [2]:
print(csqa_data["train"][0])

{'id': '075e483d21c29a511267ef62bedc0461', 'question': 'The sanctions against the school were a punishing blow, and they seemed to what the efforts the school had made to change?', 'question_concept': 'punishing', 'choices': {'label': ['A', 'B', 'C', 'D', 'E'], 'text': ['ignore', 'enforce', 'authoritarian', 'yell at', 'avoid']}, 'answerKey': 'A'}


In [3]:
print(csqa_data["train"].column_names)

['id', 'question', 'question_concept', 'choices', 'answerKey']


In [4]:
print(csqa_data)

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
        num_rows: 9741
    })
    validation: Dataset({
        features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
        num_rows: 1221
    })
    test: Dataset({
        features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
        num_rows: 1140
    })
})


In [5]:
print(csqa_data["train"][0])

{'id': '075e483d21c29a511267ef62bedc0461', 'question': 'The sanctions against the school were a punishing blow, and they seemed to what the efforts the school had made to change?', 'question_concept': 'punishing', 'choices': {'label': ['A', 'B', 'C', 'D', 'E'], 'text': ['ignore', 'enforce', 'authoritarian', 'yell at', 'avoid']}, 'answerKey': 'A'}


In [6]:
print(csqa_data["train"].column_names)

['id', 'question', 'question_concept', 'choices', 'answerKey']


In [7]:
row = csqa_data["train"][0]

choices = row["choices"]["text"]
labels = row["choices"]["label"]

correct_index = labels.index(row["answerKey"])
correct_answer = choices[correct_index]

print("ID:", row["id"])
print("Question:", row["question"])
print("Choices:", choices)
print("Correct:", correct_answer)

ID: 075e483d21c29a511267ef62bedc0461
Question: The sanctions against the school were a punishing blow, and they seemed to what the efforts the school had made to change?
Choices: ['ignore', 'enforce', 'authoritarian', 'yell at', 'avoid']
Correct: ignore


In [9]:
for dataset_name, dataset in [
    ("train", csqa_data["train"]),
    ("validation", csqa_data["validation"]),
    ("test", csqa_data["test"])
]:
    print("\nChecking:", dataset_name)

    missing = 0

    for i, row in enumerate(dataset):
        if not row["answerKey"]:
            print("Missing answerKey")
            print("Index:", i)
            print("ID:", row["id"])
            print("Question:", row["question"])
            print("AnswerKey:", repr(row["answerKey"]))
            print()

            missing += 1

    print("Total missing:", missing)


Checking: train
Total missing: 0

Checking: validation
Total missing: 0

Checking: test
Missing answerKey
Index: 0
ID: 90b30172e645ff91f7171a048582eb8b
Question: The townhouse was a hard sell for the realtor, it was right next to a high rise what?
AnswerKey: ''

Missing answerKey
Index: 1
ID: 000990552527b1353f98f1e1a7dfc643
Question: There is a star at the center of what group of celestial bodies?
AnswerKey: ''

Missing answerKey
Index: 2
ID: dca0f2859f3c3dd43a9b2bfeff4936a8
Question: What were the kids doing as they looked up at the sky and clouds?
AnswerKey: ''

Missing answerKey
Index: 3
ID: 8795a949b39702af0e452c9e1229046d
Question: The person taught an advanced class only for who?
AnswerKey: ''

Missing answerKey
Index: 4
ID: 1f74ea1f73b9f5d91a665b4d90218a6e
Question: What is a likely consequence of ignorance of rules?
AnswerKey: ''

Missing answerKey
Index: 5
ID: 0b7734f608c188350573247e3ef2a00d
Question: After graduating the dentist set up his dental office back where he grew 

In [10]:
for dataset_name, dataset in [
    ("train", csqa_data["train"]),
    ("validation", csqa_data["validation"])
]:
    print("Checking:", dataset_name)

    for i, row in enumerate(dataset):
        labels = row["choices"]["label"]
        choices = row["choices"]["text"]

        correct_index = labels.index(row["answerKey"])

        if len(choices) != 5:
            print("Wrong number of choices:", i, choices)

        if correct_index >= len(choices):
            print("Invalid correct index:", i, row["answerKey"])

    print("Done:", dataset_name)

Checking: train
Done: train
Checking: validation
Done: validation


In [11]:
import json

def normalize_csqa_row(row, include_answer=True):
    labels = row["choices"]["label"]
    choices = row["choices"]["text"]

    result = {
        "id": row["id"],
        "context": None,
        "question": row["question"],
        "choices": choices,
    }

    if include_answer:
        correct_index = labels.index(row["answerKey"])
        result["correct"] = choices[correct_index]
    else:
        result["correct"] = None

    return result


# Train
train_data = [
    normalize_csqa_row(row, include_answer=True)
    for row in csqa_data["train"]
]

# Validation
validation_data = [
    normalize_csqa_row(row, include_answer=True)
    for row in csqa_data["validation"]
]

# Test — no answerKey
test_data = [
    normalize_csqa_row(row, include_answer=False)
    for row in csqa_data["test"]
]

In [12]:
import os

output_dir = "CommonsenseQA"
os.makedirs(output_dir, exist_ok=True)

with open(os.path.join(output_dir, "train.json"), "w", encoding="utf-8") as f:
    json.dump(train_data, f, indent=2, ensure_ascii=False)

with open(os.path.join(output_dir, "validation.json"), "w", encoding="utf-8") as f:
    json.dump(validation_data, f, indent=2, ensure_ascii=False)

with open(os.path.join(output_dir, "test.json"), "w", encoding="utf-8") as f:
    json.dump(test_data, f, indent=2, ensure_ascii=False)

print("Saved to:", os.path.abspath(output_dir))

Saved to: /workspace/rehancot/CommonsenseQA
